# BERTopic — Evoluția discursului Nicușor Dan

Rulează pe **Google Colab cu T4 GPU** (free tier).

**Setări obligatorii:** Runtime → Change runtime type → **T4 GPU**

**Pașii:**
1. Verifică GPU
2. Clone repo + install deps (~2 min)
3. Embed + topic model pentru **overall** / **scris** / **vorbit** (~3 min total cu T4)
4. Zip + download `results/` (sau push înapoi pe GitHub)

**Output:** `results/03_bertopic_{overall,scris,vorbit}/` cu `topics.csv`, `topic_info.md`, `topics_over_time.png`, model serializat.

## 1. Verifică GPU

In [ ]:
!nvidia-smi | head -20

## 2. Clone repo + install deps

Repo e public, nu trebuie auth.

In [ ]:
!git clone https://github.com/radusqrt/evolutie-discurs-nicusor-dan.git
%cd evolutie-discurs-nicusor-dan

In [ ]:
!pip install -q bertopic sentence-transformers python-frontmatter stopwordsiso spacy umap-learn hdbscan
!python -m spacy download ro_core_news_sm

## 3. Rulează BERTopic pe cele 3 proiecții

Cu T4 GPU (15GB VRAM) putem rula `multilingual-e5-large` în fp32 fără probleme. Fiecare proiecție: ~1-2 min.

In [ ]:
import os
import subprocess

for projection in ["overall", "scris", "vorbit"]:
    print(f"\n{'='*70}\n  BERTopic — {projection}\n{'='*70}\n")
    env = {**os.environ, "PROJECTION": projection, "PYTHONUNBUFFERED": "1"}
    result = subprocess.run(
        ["python", "scripts/03_bertopic.py"],
        cwd=".", env=env, capture_output=False, text=True
    )
    if result.returncode != 0:
        print(f"FAILED for {projection}, exit code {result.returncode}")
        break
    else:
        print(f"\n✓ {projection} done")

## 4. Inspect quick — vezi câte topice am descoperit

In [ ]:
import pandas as pd
from pathlib import Path

for projection in ["overall", "scris", "vorbit"]:
    p = Path(f"results/03_bertopic_{projection}/topics.csv")
    if not p.exists():
        print(f"{projection}: MISSING")
        continue
    df = pd.read_csv(p)
    n_topics = df[df['topic_id'] != -1]['topic_id'].nunique()
    n_outliers = (df['topic_id'] == -1).sum()
    print(f"{projection:>8}: {len(df)} docs → {n_topics} topice + {n_outliers} outliers")

## 5. Vezi topicele descoperite (overall)

In [ ]:
from IPython.display import Markdown
Markdown(Path("results/03_bertopic_overall/topic_info.md").read_text())

## 6. Heatmap topic × perioadă (overall)

In [ ]:
from IPython.display import Image
Image("results/03_bertopic_overall/topics_over_time.png")

## 7. Zip rezultate + descarcă

Rulează cell-ul ăsta, apoi `bertopic_results.zip` va apărea în panoul Files (stânga). Click-dreapta → Download.

In [ ]:
!zip -r bertopic_results.zip results/03_bertopic_overall results/03_bertopic_scris results/03_bertopic_vorbit
from google.colab import files
files.download('bertopic_results.zip')

## (Alternativă) Push înapoi pe GitHub direct din Colab

Necesită Personal Access Token. Skip dacă preferi download local.

In [ ]:
# Uncomment + adaugă tokenul tău
# !git config user.email "radu.stochitoiu@gmail.com"
# !git config user.name "Radu Stochitoiu"
# !git add results/03_bertopic_overall results/03_bertopic_scris results/03_bertopic_vorbit
# !git commit -m "BERTopic — rezultate pe overall/scris/vorbit (rulat în Colab T4)"
# !git push https://<USERNAME>:<TOKEN>@github.com/radusqrt/evolutie-discurs-nicusor-dan.git main